In [ ]:
import ast

class Silly(ast.NodeTransformer):
    def visit_Call(self, node):
        match node:
            case ast.Call(func=ast.Name(id="silly")):
                return ast.Constant(value=48)
            case _:
                return self.generic_visit(node)

ip = get_ipython()
ip.ast_transformers.append(Silly())

In [ ]:
from functools import singledispatch

@singledispatch
def And(x,y):
    return x() and y()



In [5]:
import ast
def thunk(a : ast.Expr):
    return ast.Lambda(args=ast.arguments(posonlyargs=[], args=[], kwonlyargs=[], kw_defaults=[], defaults=[]), body=a)
class Thunk(ast.NodeTransformer):
    def visit_Call(self, node):
        match node:
            case ast.Call(func=ast.Name(id="thunk"), args=[a]):
                return thunk(self.generic_visit(a))
            case _:
                return self.generic_visit(node)

ip = get_ipython()
ip.ast_transformers.append(Thunk())      

In [ ]:
a = 4
(a := 3)
thunk(a := 3)
a

3

In [ ]:
import ast

import functools
def thunk(a : ast.Expr):
    return ast.Lambda(args=ast.arguments(posonlyargs=[], args=[], kwonlyargs=[], kw_defaults=[], defaults=[]), body=a)

from functools import singledispatch

@singledispatch
def And(x,y):
    return x and y

class OverloadAnd(ast.NodeTransformer):
    def visit_Module(self, node):
        self.active = False
        return self.generic_visit(node)
    
    def visit_Call(self, node):
        match node:
            case ast.Call(func=ast.Name(id="Expr"), args=[e]):
                self.active = True # += 1
                #print("active")
                try:
                    return self.visit(e)
                finally:
                    #print("inactive")
                    self.active = False # -= 1
    def visit_BoolOp(self, node):
        #print("visit_BoolOp", node)
        #print("active", self.active)
        if self.active:
            #print("boolop active")
            match node.op:
                case ast.And():
                    args = [self.visit(value) for value in node.values] #[thunk(self.visit(value)) for value in node.values]
                    #print([ast.dump(a) for a in args])
                    res = functools.reduce(lambda x,y: ast.Call(func=ast.Name(id="And", ctx=ast.Load()), args=[x,y], keywords=[]), args)
                    #print("res", ast.dump(res))
                    return res
        return self.generic_visit(node)
ip = get_ipython()
ip.ast_transformers.append(OverloadAnd())    

visit_BoolOp <ast.BoolOp object at 0x7f05f730f4d0>
active False


TypeError: required field "elt" missing from ListComp

In [6]:
@And.register
def _(x: int, y: int):
    return x + y
@And.register
def _(x: bool, y: bool):
    return x and y

assert Expr(True and False) is False
assert Expr(3 and 4) == 7


visit_BoolOp <ast.BoolOp object at 0x7f05f7313810>
active False
active
visit_BoolOp <ast.BoolOp object at 0x7f05f73134d0>
active True
boolop active
['Constant(value=True)', 'Constant(value=False)']
res Call(func=Name(id='And', ctx=Load()), args=[Constant(value=True), Constant(value=False)], keywords=[])
inactive
active
visit_BoolOp <ast.BoolOp object at 0x7f05f7313090>
active True
boolop active
['Constant(value=3)', 'Constant(value=4)']
res Call(func=Name(id='And', ctx=Load()), args=[Constant(value=3), Constant(value=4)], keywords=[])
inactive


Overloadable notation

In [ ]:
def sep(Collection, f)

In [1]:
True and False

from functools import singledispatch

@singledispatch
def And(x,y):
    return x and y
And(True, False)

False

In [4]:
import ast
ast.dump(ast.parse("And(True, False)"))

"Module(body=[Expr(value=Call(func=Name(id='And', ctx=Load()), args=[Constant(value=True), Constant(value=False)], keywords=[]))], type_ignores=[])"

In [ ]:
@singledispatch
def Or(x,y):
    return x() or y()
def 

Do notation
Overloadable and/if/ automatic return
auto thunking
Algerbaic effects




In [ ]:
def silly():
    """silly is a macro that expands to 48. Not actually called"""
    # Actually making the function makes the linter less mad
    raise ValueError("silly is stub")

48

In [6]:
silly(print("hello"))

48

In [1]:
import ast
class Silly(ast.NodeTransformer):
    def visit_Module(self, node):
        self.active = False
        return self.generic_visit(node)
    def visit_With(self, node):
        match node:
            case ast.With(items=[ast.withitem(context_expr=ast.Name(id="silly"))]):
                self.active = True
                try:
                    print("silly is active")
                    res = self.generic_visit(node)
                finally:
                    self.active = False
                return res
            case _:
                return self.generic_visit(node)


ip = get_ipython()
ip.ast_transformers.append(Silly())

In [ ]:
"hello"

with silly:
    print("hello")



silly is active


NameError: name 'silly' is not defined

In [ ]:
def macro(transformer):
    def res(f):
        f.

Dependent types using keyword arguments as implicits
def foo():


In [16]:
import inspect
import ast
def foo(x):
    return silly()
class Silly(ast.NodeTransformer):
    def visit_Call(self, node):
        match node:
            case ast.Call(func=ast.Name(id="silly")):
                return ast.Constant(value=48)
            case _:
                return self.generic_visit(node)
source_code = inspect.getsource(foo)
syntax_tree = ast.parse(source_code)
newcode = ast.unparse(Silly().visit(syntax_tree))
print(newcode)
exec(newcode)
foo(3)

def foo(x):
    return 48


48

In [ ]:
class Cases(ast.NodeTransformer):
    def visit_Match(self, node):
        match node:
            case ast.Match(subject=subject, cases=cases):
                pass

class Quote(ast.NodeTransformer):
    def visit_Call(self, node):
        match node:
            case ast.Call(func=ast.Name(id="quote")):
                
def quote_list(e : list) -> ast.Expr:
    ast.List(elts=[quote(x) for x in e], ctx=ast.Load())



def quote(e : ast.Expr) -> ast.Expr:
    match e:
        case ast.BinOp(left=left, op=op, right=right):
            

In [ ]:
def foo(x : doesntexists): # why would they eagerly evaluate types?
    pass

NameError: name 'doesntexists' is not defined

Do notation, multishot continuations
minikanren
julia style @var
